# MIDI composer pipeline

Fine-tune a small LLM to write classical solo-piano music from natural-language composition briefs.

1. **Describe** each piece with an LLM (features + rendered audio + embedded metadata) as a composition brief.
2. **Encode** MIDI to text with `midicomposer.codec` (implemented and round-trip tested).
3. **Build the dataset**: windowing, transposition, split by work.
4. **LoRA fine-tune** a 1.5-4B model.
5. **Generate and evaluate**.

Stages 1 and 3-5 are drafts; only stage 2 is implemented in `src/` so far.

In [ ]:
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_URL = '...'  # set once the repo has a remote
    !git clone {REPO_URL} /content/midi-composer
    %cd /content/midi-composer
    !pip install -q -e '.[describe,train]' unsloth
    !apt-get -qq install fluidsynth fluid-soundfont-gm
    MIDI_DIR = '/content/drive/MyDrive/MIDI'
    DATA_DIR = '/content/drive/MyDrive/midi-composer/data'
else:
    MIDI_DIR = os.path.expanduser('~/MIDI')
    DATA_DIR = os.path.abspath('../data')
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Composition-brief descriptions (draft)

Ground the LLM with extracted features, the embedded metadata (title, composer, tempo marking), and rendered audio. Ask for three detail levels plus a section map with bar ranges, which becomes the `<|plan|>` line. Training prompts must not name the specific piece.

In [ ]:
import glob, json, mido, numpy as np, pretty_midi

def embedded_metadata(path):
    mid = mido.MidiFile(path)
    return [m.name if m.type == 'track_name' else m.text
            for m in mid.tracks[0] if m.type in ('track_name', 'text')][:6]

def extract_features(path):
    pm = pretty_midi.PrettyMIDI(path)
    notes = [n for i in pm.instruments for n in i.notes]
    downbeats = list(pm.get_downbeats()) + [pm.get_end_time()]
    per_bar = []
    for b in range(len(downbeats) - 1):
        bn = [n for n in notes if downbeats[b] <= n.start < downbeats[b + 1]]
        per_bar.append({'bar': b + 1, 'density': len(bn),
                        'mean_vel': round(float(np.mean([n.velocity for n in bn])), 1) if bn else 0,
                        'range': [min(n.pitch for n in bn), max(n.pitch for n in bn)] if bn else None})
    return {'metadata': embedded_metadata(path), 'duration_s': round(pm.get_end_time()),
            'n_bars': len(per_bar), 'per_bar': per_bar}

PROMPT = '''You are writing a commission brief for a composer. Using the audio, the metadata,
and the per-bar analysis, return JSON with:
- brief: one imperative sentence ("Write a ...").
- paragraph: style, mood, key, meter, tempo, form, texture.
- detailed: section-by-section emotional and technical progression, with approximate
  positions in the piece ("around 40% through ..."): harmony, melody, texture, dynamics, devices.
- styled_*: the same three, additionally naming the composer's style ("in the manner of Chopin's nocturnes").
- sections: [{"name": ..., "bars": [start, end]}] covering every bar.
Never name the specific piece or its catalog number.'''

def describe(path, client, model='gemini-2.5-pro'):
    wav = os.path.join(DATA_DIR, 'wav', os.path.basename(path)[:-4] + '.wav')
    os.makedirs(os.path.dirname(wav), exist_ok=True)
    !fluidsynth -ni -F {wav} -r 22050 /usr/share/sounds/sf2/FluidR3_GM.sf2 {path} > /dev/null
    audio = client.files.upload(file=wav)
    r = client.models.generate_content(
        model=model, contents=[PROMPT, json.dumps(extract_features(path)), audio],
        config={'response_mime_type': 'application/json'})
    return {'file': os.path.basename(path), **json.loads(r.text)}

# from google import genai; client = genai.Client(api_key=...)
# with open(f'{DATA_DIR}/descriptions.jsonl', 'a') as fh:
#     for p in sorted(glob.glob(f'{MIDI_DIR}/*.mid')):
#         fh.write(json.dumps(describe(p, client)) + '\n')

## 2. MIDI to text

See `src/midicomposer/codec.py` for the format. `scripts/roundtrip.py` checks fidelity over the whole collection.

In [ ]:
from midicomposer.codec import encode, decode

pm = pretty_midi.PrettyMIDI(f'{MIDI_DIR}/appass_1.mid')
text = encode(pm)
print(text[:1200])
decode(text).save(f'{DATA_DIR}/appass_1_roundtrip.mid')

## 3. Dataset (draft)

- Windows: description + plan + previous 16 bars as context, next 32 bars as the target.
- Transposition: +/-3 semitones only, clamped to the piano range (MIDI 21-108), with key labels updated.
- Split by work: every movement of a sonata goes to the same split.

In [ ]:
import random, re

def split_bars(text):
    lines = text.splitlines()
    meta = lines[1]
    bars, cur = [], []
    for line in lines[2:-1]:
        if line.startswith('<|bar|>') and cur:
            bars.append('\n'.join(cur)); cur = []
        cur.append(line)
    bars.append('\n'.join(cur))
    return meta, bars

def windows(text, plan, ctx=16, tgt=32, stride=16):
    meta, bars = split_bars(text)
    for start in range(0, len(bars), stride):
        context = bars[max(0, start - ctx):start]
        target = bars[start:start + tgt]
        end = '\n<|piece_end|>' if start + tgt >= len(bars) else ''
        head = '<|piece_start|>\n' if start == 0 else ''
        yield {'prefix': '\n'.join([meta, plan] + context), 'target': head + '\n'.join(target) + end}

def work_id(name):
    return re.sub(r'_\d+$', '', name[:-4])

# descs = [json.loads(l) for l in open(f'{DATA_DIR}/descriptions.jsonl')]
# works = sorted({work_id(d['file']) for d in descs}); random.Random(0).shuffle(works)
# test_works = set(works[:len(works) // 10])

## 4. LoRA fine-tune (draft)

Debug the format on the Mac with MLX (1.5B, 4-bit, 4k context). Do real runs on an A100 (Colab Pro or RunPod).

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

MAX_LEN = 8192
model, tok = FastLanguageModel.from_pretrained('unsloth/Qwen2.5-3B-Instruct', max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, use_gradient_checkpointing='unsloth',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])

# train_ds: Dataset with 'prompt' (description + prefix) and 'completion' (target)
trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=train_ds,
    args=SFTConfig(output_dir=f'{DATA_DIR}/../checkpoints', per_device_train_batch_size=1,
                   gradient_accumulation_steps=8, learning_rate=2e-4, num_train_epochs=2,
                   lr_scheduler_type='cosine', max_seq_length=MAX_LEN, completion_only_loss=True,
                   bf16=True, logging_steps=10, save_steps=200))
trainer.train()

## 5. Generate and evaluate (draft)

Generate the plan, then 32-bar chunks until `<|piece_end|>`. Metrics: parse rate, key/meter/length adherence, pitch-class histogram distance to held-out works, n-gram overlap with training pieces (memorization), an LLM judge comparing a re-description of the output with the original brief, and listening.

In [ ]:
FastLanguageModel.for_inference(model)

def generate(prompt, max_new_tokens=6000, temperature=0.9):
    ids = tok.apply_chat_template([{'role': 'user', 'content': prompt}], add_generation_prompt=True,
                                  return_tensors='pt').to('cuda')
    out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.95, do_sample=True)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=False)

# Chunked composition loop, then decode(text).save(...) and render with fluidsynth to listen.